# Tratamento dos dados

## Seleção:
- Triagem das imagens, remover aquelas que possam até mesmo confundir o humano

In [1]:
from pathlib import Path

import cv2

bottle_path = Path("dados/garrafa/Garrafa").resolve()
bg_path = Path("dados/bg/Background").resolve()

teste_imagem = bottle_path / "7.png"
print(f"O arquivo {teste_imagem.name} existe lá dentro? {teste_imagem.exists()}")

if teste_imagem.exists():
    image = cv2.imread(str(teste_imagem))
    image_gray = cv2.imread(
        (str(teste_imagem)),
        cv2.IMREAD_GRAYSCALE
    )
    print("\nImagens carregadas com sucesso!")
else:
    print("\n[ERRO] O arquivo não foi encontrado. Verifique o print do caminho acima.")



O arquivo 7.png existe lá dentro? True

Imagens carregadas com sucesso!


In [2]:
print(image.shape)
print(image_gray.shape)

(562, 577, 3)
(562, 577)


## Deixando em escalas de cinza

In [3]:
inputs_dir = ["dados/garrafa/Garrafa", "dados/bg/Background"]
outputs_dir = ["dados_cinza/garrafa", "dados_cinza/bg"]

for idx, input_dir in enumerate(inputs_dir):
    input_dir = Path(input_dir)
    output_dir = Path(outputs_dir[idx])

    output_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"Lendo de: {input_dir.resolve()} | Existe? {input_dir.exists()}"
    )

    for image_path in input_dir.iterdir():
        if not image_path.is_file():
            continue

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        output_path = output_dir / f"{image_path.stem}.png"

        success = cv2.imwrite(str(output_path), image)
        if success:
            print(f"Salvo com sucesso: {output_path.name}")


Lendo de: C:\Users\Aluno\Desktop\2110\bootcampia\parte2_ml_rna\visao_comp\dados\garrafa\Garrafa | Existe? True
Salvo com sucesso: 11.png
Salvo com sucesso: 12.png
Salvo com sucesso: 13.png
Salvo com sucesso: 14.png
Salvo com sucesso: 18.png
Salvo com sucesso: 20261019_102046.png
Salvo com sucesso: 20261019_102054.png
Salvo com sucesso: 20261019_102123.png
Salvo com sucesso: 20261019_102130.png
Salvo com sucesso: 20261019_102149.png
Salvo com sucesso: 20261019_102150.png
Salvo com sucesso: 20261019_102157.png
Salvo com sucesso: 20261019_102201.png
Salvo com sucesso: 20261019_102325.png
Salvo com sucesso: 20261019_102335.png
Salvo com sucesso: 20261019_102350.png
Salvo com sucesso: 20261019_102408.png
Salvo com sucesso: 20261019_102515.png
Salvo com sucesso: 20261019_102538.png
Salvo com sucesso: 20261019_102730.png
Salvo com sucesso: 20261019_103210.png
Salvo com sucesso: 20261019_103252.png
Salvo com sucesso: 20261019_103413.png
Salvo com sucesso: 20261019_103426.png
Salvo com sucesso:

In [4]:
inputs_dir = ["dados_cinza/garrafa", "dados_cinza/bg"]
outputs_dir = ["dados_rot/garrafa", "dados_rot/bg"]

for idx, input_dir in enumerate(inputs_dir):
    input_dir = Path(input_dir)
    output_dir = Path(outputs_dir[idx])

    output_dir.mkdir(parents=True, exist_ok=True)

    for image_path in input_dir.iterdir():
        if not image_path.is_file():
            continue

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        #Flip horizontal
        horizontal = cv2.flip(image, 1)

        #Flip vertical
        vertical = cv2.flip(image, 0)

        rotate_90_cc = cv2.rotate(
            image,
            cv2.ROTATE_90_COUNTERCLOCKWISE
        )

        rotate_90_cw = cv2.rotate(
            image,
            cv2.ROTATE_90_CLOCKWISE
        )

        rotate_180 = cv2.rotate(
            image,
            cv2.ROTATE_180
        )

        augmentations = {
        "flip_h": horizontal,
        "flip_v": vertical,
        "rot_90_cc": rotate_90_cc,
        "rot_90_cw": rotate_90_cw,
        "rot_180": rotate_180,
        }

        for suffix, augmented in augmentations.items():
            output_path = output_dir / (
                f"{image_path.stem}_{suffix}.png"
            )

            cv2.imwrite(str(output_path), augmented)

In [6]:
inputs_dir = ["dados_rot/garrafa", "dados_rot/bg"]
outputs_dir = ["dados_redm/garrafa", "dados_redm/bg"]

target_width = 130
target_height = 130

for idx, input_dir in enumerate(inputs_dir):
    input_dir = Path(input_dir)
    output_dir = Path(outputs_dir[idx])

    output_dir.mkdir(parents=True, exist_ok=True)

    for image_path in input_dir.iterdir():
        if not image_path.is_file():
            continue

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue

        height, width = image.shape
        scale = min(target_width / width, target_height / height)
        
        new_width = round(width * scale)
        new_height = round(height * scale)

        resized = cv2.resize(
            image,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA
        )

        result = cv2.copyMakeBorder(
            resized,
            top=(target_height - new_height) // 2,
            bottom=(target_height - new_height + 1) // 2,
            left=(target_width - new_width) // 2,
            right=(target_width - new_width + 1) // 2,
            borderType=cv2.BORDER_CONSTANT,
            value=255
        )

        output_path = output_dir / f"{image_path.stem}.png"

        success = cv2.imwrite(str(output_path), result)